In [1]:
import json, glob, os
import pandas as pd

def collect(results_root):
    rows = []
    for jf in glob.glob(os.path.join(results_root, "**", "*_results.json"), recursive=True):
        with open(jf) as f:
            data = json.load(f)
        mode = "ROI" if data.get("has_roi_metadata") else "Full"
        for q_key, q_data in data["experiments"].items():
            q = q_key.split("_")[1]
            vm = q_data.get("volume_metrics", {})
            qm = q_data.get("quality_metrics", {})
            if not vm or qm.get("success_rate", 0) == 0:
                continue
            rows.append({
                "subject": data.get("filename", ""),
                "Model": f"JPEG2000_lv{q}",
                "Mode": mode,
                "CR":        vm.get("compression_ratio_memory", 0),
                "BPP":       vm.get("bpp", 0),
                "PSNR":      qm.get("avg_psnr", 0),
                "SSIM":      qm.get("avg_ssim", 0),
                "Comp_s":    vm.get("total_compression_time", 0),
                "Decomp_s":  vm.get("total_decompression_time", 0),
            })
    return pd.DataFrame(rows)

# Gộp cả full và roi
df_full = collect("./results/jpeg2000/full")
df_roi  = collect("./results/jpeg2000/roi")
df = pd.concat([df_full, df_roi], ignore_index=True)

# Tính trung bình theo Model + Mode (giống Table 2 trong paper)
summary = df.groupby(["Model", "Mode"])[
    ["CR","BPP","PSNR","SSIM","Comp_s","Decomp_s"]
].mean().round(3)

print("\n=== KẾT QUẢ TỔNG HỢP (trung bình tất cả subjects) ===")
print(summary.to_string())
summary.to_csv("./results/jpeg2000_summary.csv")
print("\nĐã lưu: ./results/jpeg2000_summary.csv")

KeyError: 'Model'